# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [1]:
import os, tiktoken
from langchain_classic import hub
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

from IPython.display import Markdown, display

from  dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate 

import warnings
warnings.filterwarnings('ignore')

d:\ironhack_bootcamp\main-bootcamp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
from langchain_core.prompts import ChatPromptTemplate

In [4]:
# Load GPT-4 model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Tokenizer for GPT-4
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Tavily instance
tavily_search = TavilySearchResults()

# Define safe search function (trim to 3500 tokens)
def safe_search(query: str) -> str:
    """
    Performs a Tavily web search and ensures the result is short enough
    to fit within the GPT-4 token limit.
    """
    result = tavily_search.invoke({"query": query})  # Use invoke instead of run
    
    # Handle result format
    if isinstance(result, list):
        result_text = " ".join([str(item) for item in result])
    elif isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)
    
    # Count and trim tokens
    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  
    return trimmed

# Define the tool
tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Web search tool that brings lastest information and trims output after one search to avoid token overflow with GPT-4o-mini"
    )
]

# Define Prompt Mechanism
# define system prompt
system_prompt = "Use web search only if necessary. After receiving search results, answer directly without calling the tool again unless information is clearly missing."

# Get the React prompt from hub
base_prompt = hub.pull("hwchase17/react")

new_template = system_prompt + "\n\n" + base_prompt.template

prompt = PromptTemplate(
    input_variables=base_prompt.input_variables,
    template=new_template,
)

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create agent executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,
    handle_parsing_errors=True, 
    max_iterations = 8,
    max_execution_time= 120
)

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [5]:
# Your prompt here
prompt_1 = """
Summarize the latest developments in generative AI models published in 2026 in Markdown format with:
- Main heading: Latest Generative AI (2026)
- Subheadings per key model or development
- Bullet points for features
- Source links if any
Keep it concise.
"""

# Run it
response_1 = agent_executor({"input":prompt_1})

display(Markdown(response_1["output"]))

```markdown
# Latest Generative AI (2026)

## Multimodal AI Models
- **Key Development**: Generative models are evolving to be multisensory, allowing them to interpret the world similarly to humans.
- **Features**:
  - Ability to bridge language, vision, and action.
  - Development of autonomous digital workers capable of complex tasks.
  - Emergence of decentralized networks of agents for knowledge sharing and specialization.
- **Source**: [IBM News](https://www.ibm.com/think/news/ai-tech-trends-predictions-2026)

## Top Generative AI Models
- **Leading Models**: 
  - **Gemini 3 Pro** (Google)
  - **GPT-5.2** (OpenAI)
  - **Claude Opus 4.5** (Anthropic)
- **Notable Trends**:
  - Open-source models like Qwen3-Max are gaining traction.
  - Size limitations of models affecting their accessibility.
- **Source**: [VirtusLab](https://virtuslab.com/blog/ai/best-gen-ai-beginning-2026/)

## AI in Code Generation
- **Development**: OpenAI's GPT-5.3-Codex is designed for generating code and has been instrumental in its own development.
- **Features**:
  - Recursive AI capabilities for self-improvement.
  - Enhanced quality in AI outputs and new modalities beyond text.
- **Source**: [AI and Academia](https://aiandacademia.substack.com/p/ai-technology-developments-in-early)

## Trends in AI Utilization
- **Focus Shift**: Generative AI is moving from individual use to organizational resource management.
- **Key Observations**:
  - Addressing the value-realization problem of generative AI.
  - Increased integration of AI into enterprise-level applications.
- **Source**: [MIT Sloan Management Review](https://sloanreview.mit.edu/article/five-trends-in-ai-and-data-science-for-2026/)

## Healthcare Applications
- **Advancements**: Generative AI is expanding into healthcare, aiding in diagnostics and treatment planning.
- **Impact**: New products and services are becoming available to a broader audience, addressing global healthcare access issues.
- **Source**: [Microsoft News](https://news.microsoft.com/source/features/ai/whats-next-in-ai-7-trends-to-watch-in-2026/)
```
This summary captures the latest developments in generative AI models for 2026, highlighting key models, trends, and applications.

### Exercise 2: AI Startups Landscape

Goal: Track 2026’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [6]:
# Your prompt here
prompt_2 = """ 
Summerize top3 emerging AI startup in 2026 and their innovation in matkdown format. 
- Main heading: Startup Name
- Add sub-headings: About company
- Undersub-heading add 2-3 bullets points explaining what company does and core innvation.
- Source links if any 
Keep it concise."""

# run the agent and get the response
response_2 = agent_executor({"input":prompt_2})

In [7]:
display(Markdown(response_2["output"]))

### Humane
#### About the company
- Humane is focused on creating AI-powered wearable devices that integrate seamlessly with human interaction.
- Their innovation lies in merging technology with everyday life, enhancing user experience through intuitive design.

### ElevenLabs
#### About the company
- ElevenLabs is a voice-AI pioneer that has achieved significant revenue growth, reaching approximately $200M in annual recurring revenue.
- They specialize in voice synthesis technology, enabling high-quality voice generation for various applications, including enterprise and creative sectors.

### Synthesia
#### About the company
- Synthesia is a leader in generative video technology, allowing users to create studio-quality AI-generated videos.
- Their platform has surpassed $100M in annual recurring revenue, showcasing the demand for scalable AI-driven content creation tools.

### Sources
- [LinkedIn Article on AI Startups](https://www.linkedin.com/posts/rathanuday_every-few-years-a-new-generation-of-startups-activity-7393717758343114752-eKnh)
- [Linas's Newsletter on AI Startups](https://linas.substack.com/p/top10aistartups2026)
- [eLearning Industry on Hottest AI Startups](https://elearningindustry.com/advertise/elearning-marketing-resources/blog/hottest-ai-startups-the-companies-defining-the-next-wave-of-ai)

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [8]:
# Your prompt here
prompt_3 = """
Compare OpenAIs ChatGPT 5.2 and Anthropics Claude Pro in Markdown with:

- Main heading: Product Comparison (ChatGPT 5.2 vs Claude Pro)
- Subheadings per product 
- Bullet points for key features, pricing, performance
- Source links if available
Keep it concise.
"""
# Run it
response_3 = agent_executor({"input": prompt_3})

In [9]:
display(Markdown(response_3['output']))

```markdown
# Product Comparison (ChatGPT 5.2 vs Claude Pro)

## OpenAI's ChatGPT 5.2
- **Key Features:**
  - Pricing: $1.75 per million input tokens, $14 per million output tokens.
  - Token efficiency: Uses approximately 50% fewer tokens than competitors for similar quality.
  - Context window: 128K tokens.
  - Best for: Coding, agentic tasks, and office productivity.
  - Performance: Reduced hallucinations by 30%, saves enterprise users 40–60 minutes daily on business tasks.

- **Pricing:**
  - Subscription plans start at $20 monthly for Plus.
  - For 10 million output tokens, costs approximately $140.

- **Performance:**
  - 80% on SWE-bench for coding tasks.
  - Designed for high efficiency in general-purpose tasks.

[Source](https://openai.com/index/introducing-gpt-5-2/)

## Anthropic's Claude Pro
- **Key Features:**
  - Pricing: $5 per million input tokens, $25 per million output tokens.
  - Token efficiency: Uses 76% fewer tokens while matching performance of competitors.
  - Context window: 200K tokens (1M beta).
  - Best for: Complex reasoning, content generation, and production-grade code.

- **Pricing:**
  - For 10 million output tokens, costs around $250.

- **Performance:**
  - 80.9% on SWE-bench for software engineering tasks.
  - Industry-leading robustness against prompt injection attacks.

[Source](https://www.jenova.ai/en/resources/ai-model-comparison-tool)
```
This comparison highlights the key features, pricing, and performance of both models in a concise manner.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


In [10]:
custom_prompt = """
Who is top3 Amazon's competitor and which strategy they have implemented in last 6 months.
Give the result in Markdown as final answer:
- Main heading: Competitor Name
- Add 2-3 small bullet points as strategy per competitor.
- Add source links if available
Keep it concise and factual.
"""
response_custom = agent_executor.invoke({"input": custom_prompt})

In [11]:
display(Markdown(response_custom['output']))

```markdown
# Competitor Name: Walmart
- Omnichannel Strategy: Enhanced "stores as hubs" for same-day delivery.
- E-commerce Growth: Revenue of $648 billion in 2023, surpassing Amazon.
- Sustainability Initiatives: Focus on eco-friendly products and carbon footprint reduction.

# Competitor Name: Target
- Curated Merchandise: Emphasis on exclusive collaborations and trendy products.
- Digital Fulfillment Expansion: Same-day delivery and order pickup services.
- Loyalty Programs: Enhanced Target REDcard loyalty program.

# Competitor Name: JD.com
- Logistics Optimization: Fast delivery and customer service focus.
- B2C Model: Strengthening direct sales to consumers.
- Revenue Growth: 6.8% increase in net revenue, reaching $158.8 billion.
```

In [12]:
prompt4= """ Why there was strike on 01.03.2026 in germany? Attach article source.
"""
response4 = agent_executor.invoke({"input": prompt4})
                                        
display(Markdown(response4['output']))

The strike on March 1, 2026, in Germany was due to a nationwide public transport strike initiated by the Verdi Union over stalled wage negotiations and demands for better working conditions. More details can be found in the article from DW.com [here](https://www.dw.com/en/germany-braces-for-more-strikes-in-2026/a-75937430).

In [13]:
prompt5= "Why there was strike yesterday in germany? Attach article source."
response5 = agent_executor.invoke({"input": prompt5})
                                        
display(Markdown(response5['output']))

The strike in Germany yesterday was part of ongoing actions by public sector workers who were negotiating for better wages and working conditions. A significant wage deal was reached, providing an 11% pay hike for employees of Germany's 16 states. For more details, you can read the article from DW.com [here](https://www.dw.com/en/germany-public-sector-workers-strike-wage-deal/a-67678397).

In [14]:
prompt6 = "Who won the yesterday cricket match?"

response6 = agent_executor.invoke({"input": prompt6})
                                        
display(Markdown(response6['output']))

Australia won the cricket match against the Netherlands by 309 runs on October 25, 2023.

In [15]:
prompt7 = "Who won cricket match played on 01.03.2026?"

response7 = agent_executor.invoke({"input": prompt7})
                                        
display(Markdown(response7['output']))

India won the cricket match played on March 1, 2026, against West Indies by five wickets.